In [ ]:
# In[1]:

import pandas as pd
import pytz

# reuse variable names where possible
tz = pytz.timezone('Asia/Shanghai')

# load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# convert timestamp ints to timezone-aware datetimes in UTC+8
metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Metric: group by (cmdb_id, kpi_name) and compute count, min_ts, max_ts, global P95 of value
metric_grp = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)
# convert min/max timestamps to tz-aware datetimes for readability
metric_grp['min_ts_dt'] = pd.to_datetime(metric_grp['min_ts'], unit='s', utc=True).dt.tz_convert(tz)
metric_grp['max_ts_dt'] = pd.to_datetime(metric_grp['max_ts'], unit='s', utc=True).dt.tz_convert(tz)

metric_summary = metric_grp.sort_values('count', ascending=False).head(20)[
    ['cmdb_id', 'kpi_name', 'count', 'min_ts_dt', 'max_ts_dt', 'p95_value']
]

# distinct cmdb_id values and distinct kpi_name values with counts (compact)
metric_cmdb_counts = metric_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_counts = metric_df['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count')

# Trace: group by (cmdb_id, trace_name) and compute count, min_ts, max_ts, global P95 of value
trace_grp = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)
trace_grp['min_ts_dt'] = pd.to_datetime(trace_grp['min_ts'], unit='s', utc=True).dt.tz_convert(tz)
trace_grp['max_ts_dt'] = pd.to_datetime(trace_grp['max_ts'], unit='s', utc=True).dt.tz_convert(tz)

trace_summary = trace_grp.sort_values('count', ascending=False).head(20)[
    ['cmdb_id', 'trace_name', 'count', 'min_ts_dt', 'max_ts_dt', 'p95_value']
]

trace_cmdb_counts = trace_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
trace_tracename_counts = trace_df['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='count')

# overall min and max timestamp across both files (as timezone-aware datetimes)
overall_min_ts = min(metric_df['timestamp'].min() if not metric_df.empty else None,
                     trace_df['timestamp'].min() if not trace_df.empty else None)
overall_max_ts = max(metric_df['timestamp'].max() if not metric_df.empty else None,
                     trace_df['timestamp'].max() if not trace_df.empty else None)
overall_time_range = {
    'overall_min_ts_dt': pd.to_datetime(overall_min_ts, unit='s', utc=True).tz_convert(tz) if overall_min_ts is not None else None,
    'overall_max_ts_dt': pd.to_datetime(overall_max_ts, unit='s', utc=True).tz_convert(tz) if overall_max_ts is not None else None
}

# Final compact outputs (variables shown, not full raw frames)
metric_summary, metric_cmdb_counts, metric_kpi_counts, trace_summary, trace_cmdb_counts, trace_tracename_counts, overall_time_range

```
Out[1]:
```
```python
# Compose a plain-English summary string based on prior results in the kernel
num_metric_groups = metric_grp.shape[0] if 'metric_grp' in globals() else None
num_metric_cmdb = metric_cmdb_counts.shape[0] if 'metric_cmdb_counts' in globals() else None
num_metric_kpi = metric_kpi_counts.shape[0] if 'metric_kpi_counts' in globals() else None
top_metric_count = int(metric_summary['count'].max()) if 'metric_summary' in globals() else None
top_metric_examples = metric_summary[['cmdb_id','kpi_name','count']].head(3).to_dict(orient='records') if 'metric_summary' in globals() else []

num_trace_groups = trace_grp.shape[0] if 'trace_grp' in globals() else None
num_trace_cmdb = trace_cmdb_counts.shape[0] if 'trace_cmdb_counts' in globals() else None
num_trace_names = trace_tracename_counts.shape[0] if 'trace_tracename_counts' in globals() else None
top_trace_count = int(trace_summary['count'].max()) if 'trace_summary' in globals() else None
top_trace_examples = trace_summary[['cmdb_id','trace_name','count']].head(3).to_dict(orient='records') if 'trace_summary' in globals() else []

overall_min = overall_time_range.get('overall_min_ts_dt') if 'overall_time_range' in globals() else None
overall_max = overall_time_range.get('overall_max_ts_dt') if 'overall_time_range' in globals() else None

summary = (
    f"Summary:\n"
    f"- metric.csv: {num_metric_groups} (cmdb_id,kpi_name) groups. "
    f"{num_metric_cmdb} distinct cmdb_id, {num_metric_kpi} distinct kpi_name. "
    f"Top group count = {top_metric_count}. Example top groups: {top_metric_examples}.\n"
    f"- trace.csv: {num_trace_groups} (cmdb_id,trace_name) groups. "
    f"{num_trace_cmdb} distinct cmdb_id, {num_trace_names} distinct trace_name. "
    f"Top group count = {top_trace_count}. Example top groups: {top_trace_examples}.\n"
    f"- Time range across both files: {overall_min} to {overall_max} (Asia/Shanghai, UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                  kpi_name  count                 min_ts_dt                 max_ts_dt    p95_value
1638   os_020              Disk_io_util     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.000000
1057   os_008                Send_total     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.000685
1256   os_012      Processor_load_5_min     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.016667
1395   os_015  Incoming_network_traffic     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.002348
1230   os_012              Disk_io_util     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     1.000000
1761   os_022  Outgoing_network_traffic     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.063762
1784  osb_001           app.succee_rate     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     1.000000
1782  osb_001                   app.num     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00   461.000000
1781  osb_001              app.avg_time     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.036211
976    os_007               Disk_rd_ios     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.000000
322    db_008              CPU_free_pct     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00    97.646099
95     db_003              CPU_free_pct     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00    99.284146
955    os_006                Send_total     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.000477
289    db_007             MEM_real_util     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00    83.520000
5      db_001              CPU_free_pct     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00    99.006543
1516   os_017                Send_total     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00     0.248793
1783  osb_001            app.succee_num     61 2020-05-30 03:00:00+08:00 2020-05-30 04:00:00+08:00   461.000000
802    os_003                Send_total     60 2020-05-30 03:00:00+08:00 2020-05-30 03:59:00+08:00     3.607865
801    os_003                Recv_total     60 2020-05-30 03:00:00+08:00 2020-05-30 03:59:00+08:00  2489.260673
816    os_004           CPU_system_time     60 2020-05-30 03:00:00+08:00 2020-05-30 03:59:00+08:00     0.591278,        cmdb_id  count
0       db_007   2108
1       db_003   2107
2       db_012   2101
3       db_013   2100
4       db_011   2098
..         ...    ...
51  docker_005    540
52  docker_003    522
53  docker_002    504
54  docker_008    477
55     osb_001    244

[56 rows x 2 columns],                      kpi_name  count
0                Disk_io_util   1321
1    Outgoing_network_traffic   1321
2                  Send_total   1320
3    Incoming_network_traffic   1319
4                 Disk_wr_kbs   1318
..                        ...    ...
127               Tbs_Free_Gb     26
128              Tbs_Used_Pct     26
129               UndoTbs_Pct     26
130              Memory_total     22
131              SEQ_Used_Pct     14

[132 rows x 2 columns],        cmdb_id                                     trace_name  count                 min_ts_dt                 max_ts_dt    p95_value
0   docker_001  trace.from_os_022.remoteprocess.duration_mean     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00    16.173160
1   docker_001   trace.from_os_022.remoteprocess.duration_p95     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00    33.262450
2   docker_001     trace.from_os_022.remoteprocess.error_rate     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.000000
3   docker_001      trace.from_os_022.remoteprocess.row_count     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00   118.100000
4   docker_001                   trace.self.csf.duration_mean     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     7.208347
5   docker_001                    trace.self.csf.duration_p95     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00    15.184525
6   docker_001                      trace.self.csf.error_rate     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.411957
7   docker_001                       trace.self.csf.row_count     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00   472.400000
8   docker_001             trace.self.flyremote.duration_mean     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.014276
9   docker_001              trace.self.flyremote.duration_p95     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.092100
10  docker_001                trace.self.flyremote.error_rate     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.000000
11  docker_001                 trace.self.flyremote.row_count     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00   118.100000
12  docker_001                  trace.self.jdbc.duration_mean     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.004329
13  docker_001                   trace.self.jdbc.duration_p95     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.008400
14  docker_001                     trace.self.jdbc.error_rate     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.000000
15  docker_001                      trace.self.jdbc.row_count     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00  1895.000000
16  docker_001                 trace.self.local.duration_mean     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.020999
17  docker_001                  trace.self.local.duration_p95     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.098550
18  docker_001                    trace.self.local.error_rate     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00     0.000000
19  docker_001                     trace.self.local.row_count     30 2020-05-30 03:30:00+08:00 2020-05-30 03:59:00+08:00  1653.400000,       cmdb_id  count
0  docker_001    840
1  docker_002    840
2  docker_003    840
3  docker_004    840
4  docker_005    480
5  docker_006    480
6  docker_007    480
7  docker_008    480
8      os_021    480
9      os_022    480,                                          trace_name  count
0                         trace.self.jdbc.row_count    240
1                        trace.self.jdbc.error_rate    240
2                      trace.self.jdbc.duration_p95    240
3                     trace.self.jdbc.duration_mean    240
4                        trace.self.local.row_count    240
..                                              ...    ...
71      trace.to_docker_001.remoteprocess.row_count     30
72  trace.to_docker_002.remoteprocess.duration_mean     30
73   trace.to_docker_002.remoteprocess.duration_p95     30
74     trace.to_docker_002.remoteprocess.error_rate     30
75      trace.to_docker_002.remoteprocess.row_count     30

[76 rows x 2 columns], {'overall_min_ts_dt': Timestamp('2020-05-30 03:00:00+0800', tz='Asia/Shanghai'), 'overall_max_ts_dt': Timestamp('2020-05-30 04:00:00+0800', tz='Asia/Shanghai')})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# reuse existing metric_df and tz if present, otherwise load
tz = pytz.timezone('Asia/Shanghai')
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
    metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Calculate global P95 and P5 per (cmdb_id, kpi_name) using full series (before any filtering)
group_thresh = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p95=lambda s: s.quantile(0.95), p5=lambda s: s.quantile(0.05))
)

# 2) Merge thresholds back to full metric series
metric_with_thresh = metric_df.merge(group_thresh, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Define incident window in Asia/Shanghai (UTC+8) and filter rows into window (inclusive)
start = pd.Timestamp('2020-05-30 03:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-30 04:00:00').tz_localize(tz)
in_window = metric_with_thresh[(metric_with_thresh['ts_dt'] >= start) & (metric_with_thresh['ts_dt'] <= end)].copy()

# 4) Detect anomaly points in window where value >= global_P95 OR value <= global_P5
anoms = in_window[(in_window['value'] >= in_window['p95']) | (in_window['value'] <= in_window['p5'])].copy()

# 5) Compute breach severity per anomaly row:
#    high_sev = (value - p95)/p95  (if p95==0 and value>p95 -> inf)
#    low_sev  = (p5 - value)/p5    (if p5==0 and value<p5 -> inf)
if not anoms.empty:
    high_sev = pd.Series(np.nan, index=anoms.index)
    low_sev = pd.Series(np.nan, index=anoms.index)

    mask_high = anoms['value'] >= anoms['p95']
    denom_high = anoms.loc[mask_high, 'p95']
    num_high = anoms.loc[mask_high, 'value'] - anoms.loc[mask_high, 'p95']
    # safe division, handle zero denom
    high_sev.loc[mask_high] = num_high / denom_high.replace(0, np.nan)
    zero_den_high = mask_high & (anoms['p95'] == 0) & (anoms['value'] > anoms['p95'])
    high_sev.loc[zero_den_high] = np.inf

    mask_low = anoms['value'] <= anoms['p5']
    denom_low = anoms.loc[mask_low, 'p5']
    num_low = anoms.loc[mask_low, 'p5'] - anoms.loc[mask_low, 'value']
    low_sev.loc[mask_low] = num_low / denom_low.replace(0, np.nan)
    zero_den_low = mask_low & (anoms['p5'] == 0) & (anoms['value'] < anoms['p5'])
    low_sev.loc[zero_den_low] = np.inf

    # max magnitude severity (choose larger of high_sev and low_sev where applicable)
    anoms['severity'] = pd.concat([high_sev.fillna(-np.inf), low_sev.fillna(-np.inf)], axis=1).max(axis=1)
    # replace -inf back to NaN for rows that had neither (shouldn't happen)
    anoms.loc[anoms['severity'] == -np.inf, 'severity'] = np.nan
else:
    anoms['severity'] = []

# 6) Aggregate anomalies per (cmdb_id, kpi_name)
if not anoms.empty:
    anomaly_summary = (
        anoms
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anom_ts=('ts_dt', 'min'),
            max_breach_severity=('severity', 'max')
        )
        .sort_values('anomaly_count', ascending=False)
    )
    # keep compact: round severity
    anomaly_summary['max_breach_severity'] = anomaly_summary['max_breach_severity'].replace(np.inf, np.inf)
    anomaly_summary['max_breach_severity'] = anomaly_summary['max_breach_severity'].round(6)
else:
    anomaly_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anom_ts','max_breach_severity'])

# 7) Top 20 rows sorted by anomaly_count desc, and single-line summary count
top_anomalies = anomaly_summary.sort_values('anomaly_count', ascending=False).head(20)
distinct_count_line = f"{anomaly_summary.shape[0]} distinct (cmdb_id, kpi_name) had anomalies in the window."

# Final compact outputs
top_anomalies[['cmdb_id','kpi_name','anomaly_count','earliest_anom_ts','max_breach_severity']], distinct_count_line

```
Out[2]:
```
```python
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse top_anomalies and anomaly_summary from the kernel
count_distinct = anomaly_summary.shape[0] if 'anomaly_summary' in globals() else 0

# prepare top 5 examples
top5 = top_anomalies.head(5) if 'top_anomalies' in globals() else None
lines = []
if top5 is not None and not top5.empty:
    for i, row in enumerate(top5.itertuples(index=False), 1):
        cmdb = row.cmdb_id
        kpi = row.kpi_name
        cnt = int(row.anomaly_count)
        earliest = row.earliest_anom_ts.tz_convert(tz).strftime('%Y-05-30 %H:%M:%S%z') if hasattr(row.earliest_anom_ts, 'tzinfo') else str(row.earliest_anom_ts)
        sev = row.max_breach_severity
        if pd.isna(sev):
            sev_s = 'N/A'
        elif np.isinf(sev):
            sev_s = 'inf'
        else:
            sev_s = f"{float(sev):.6f}"
        lines.append(f"{i}) {cmdb}.{kpi} — count={cnt}, earliest={earliest}, max_severity={sev_s}")
else:
    lines.append("No anomalies found in the window.")

summary = (
    f"Summary (plain English):\n"
    f"- Incident window: 2020-05-30 03:30:00+08 to 2020-05-30 04:00:00+08 (inclusive).\n"
    f"- Distinct (cmdb_id, kpi_name) with anomalies: {count_distinct}.\n"
    f"- Top anomalies by count (up to 5 shown):\n" + "\n".join(lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id             kpi_name  anomaly_count          earliest_anom_ts  max_breach_severity
946      os_007          Disk_rd_ios             31 2020-05-30 03:30:00+08:00                  NaN
1585     os_020         Disk_io_util             31 2020-05-30 03:30:00+08:00                  NaN
1949  redis_011      Redis_key_count             30 2020-05-30 03:30:00+08:00             0.000000
1637     os_021          Disk_rd_ios             30 2020-05-30 03:30:00+08:00                  NaN
1506     os_018              Page_pi             30 2020-05-30 03:30:00+08:00                  inf
1486     os_018         Disk_io_util             30 2020-05-30 03:30:00+08:00                  NaN
1487     os_018          Disk_rd_ios             30 2020-05-30 03:30:00+08:00                  NaN
1488     os_018          Disk_rd_kbs             30 2020-05-30 03:30:00+08:00                  inf
1596     os_020            ICMP_ping             30 2020-05-30 03:30:00+08:00             0.000000
1589     os_020          Disk_wr_ios             30 2020-05-30 03:30:00+08:00             1.333333
1586     os_020          Disk_rd_ios             30 2020-05-30 03:30:00+08:00                  NaN
542      db_013            MEM_Total             30 2020-05-30 03:30:00+08:00             0.000000
537      db_013                 Hang             30 2020-05-30 03:30:00+08:00             0.000000
526      db_012  tnsping_result_time             30 2020-05-30 03:30:00+08:00             0.000000
527      db_013                  ACS             30 2020-05-30 03:30:00+08:00             1.000000
528      db_013                 AIOS             30 2020-05-30 03:30:00+08:00                  NaN
259      db_006  tnsping_result_time             30 2020-05-30 03:30:00+08:00             0.000000
510      db_012             Row_Lock             30 2020-05-30 03:30:00+08:00                  NaN
511      db_012      SctRead_Per_Sec             30 2020-05-30 03:30:00+08:00                  NaN
1639     os_021           Disk_svctm             30 2020-05-30 03:30:00+08:00                  NaN, '1993 distinct (cmdb_id, kpi_name) had anomalies in the window.')```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# reuse existing trace_df and tz if present, otherwise load and prepare
tz = pytz.timezone('Asia/Shanghai')
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')
if 'ts_dt' not in trace_df.columns:
    trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series
trace_thresh = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda s: s.quantile(0.95), p5=lambda s: s.quantile(0.05))
)

# 2) Merge thresholds back to trace series
trace_with_thresh = trace_df.merge(trace_thresh, on=['cmdb_id', 'trace_name'], how='left')

# 3) Define incident window (inclusive) in Asia/Shanghai and filter
start = pd.Timestamp('2020-05-30 03:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-30 04:00:00').tz_localize(tz)
trace_window = trace_with_thresh[(trace_with_thresh['ts_dt'] >= start) & (trace_with_thresh['ts_dt'] <= end)].copy()

# 4) Detect anomalies in window where value >= p95 OR value <= p5
anoms = trace_window[(trace_window['value'] >= trace_window['p95']) | (trace_window['value'] <= trace_window['p5'])].copy()

# 5) Compute per-row breach severity (handle zero thresholds -> inf if breached)
if not anoms.empty:
    # prepare series
    high_sev = pd.Series(np.nan, index=anoms.index)
    low_sev = pd.Series(np.nan, index=anoms.index)

    mask_high = anoms['value'] >= anoms['p95']
    # compute safely for non-zero p95
    denom_high = anoms.loc[mask_high, 'p95']
    num_high = anoms.loc[mask_high, 'value'] - anoms.loc[mask_high, 'p95']
    high_sev.loc[mask_high] = num_high / denom_high.replace(0, np.nan)
    # if p95==0 and value>p95 => infinite relative breach
    zero_den_high = mask_high & (anoms['p95'] == 0) & (anoms['value'] > anoms['p95'])
    high_sev.loc[zero_den_high] = np.inf

    mask_low = anoms['value'] <= anoms['p5']
    denom_low = anoms.loc[mask_low, 'p5']
    num_low = anoms.loc[mask_low, 'p5'] - anoms.loc[mask_low, 'value']
    low_sev.loc[mask_low] = num_low / denom_low.replace(0, np.nan)
    zero_den_low = mask_low & (anoms['p5'] == 0) & (anoms['value'] < anoms['p5'])
    low_sev.loc[zero_den_low] = np.inf

    # choose larger magnitude severity per row
    anoms['severity'] = pd.concat([high_sev.fillna(-np.inf), low_sev.fillna(-np.inf)], axis=1).max(axis=1)
    anoms.loc[anoms['severity'] == -np.inf, 'severity'] = np.nan
else:
    anoms['severity'] = pd.Series(dtype=float)

# 6) Aggregate anomalies per cmdb_id
if not anoms.empty:
    cmdb_agg = (
        anoms
        .groupby('cmdb_id', as_index=False)
        .agg(
            total_anomaly_count=('value', 'size'),
            earliest_anomaly_ts=('ts_dt', 'min'),
            max_breach_severity=('severity', 'max')
        )
        .sort_values('total_anomaly_count', ascending=False)
    )
    # round severity for compactness, keep inf as is
    cmdb_agg['max_breach_severity'] = cmdb_agg['max_breach_severity'].replace(np.inf, np.inf)
    cmdb_agg['max_breach_severity'] = cmdb_agg['max_breach_severity'].round(6)
else:
    cmdb_agg = pd.DataFrame(columns=['cmdb_id','total_anomaly_count','earliest_anomaly_ts','max_breach_severity'])

# 7) Top 20 cmdb_id rows and distinct count of cmdb_id with anomalies
top_trace_cmdb_anomalies = cmdb_agg.head(20)[['cmdb_id','total_anomaly_count','earliest_anomaly_ts','max_breach_severity']]
distinct_cmdb_count = int(cmdb_agg.shape[0])

# Return compact outputs
top_trace_cmdb_anomalies, distinct_cmdb_count

```
Out[3]:
```
```python
# Compose a plain-English summary string from prior results in the kernel
import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# reuse variables from previous execution
top_df = top_trace_cmdb_anomalies if 'top_trace_cmdb_anomalies' in globals() else None
distinct_count = distinct_cmdb_count if 'distinct_cmdb_count' in globals() else 0

lines = []
lines.append("Trace anomaly summary (window 2020-05-30 03:30:00+08 to 2020-05-30 04:00:00+08):")
lines.append(f"- Distinct cmdb_id with trace anomalies: {distinct_count}")

if top_df is not None and not top_df.empty:
    lines.append("- Top cmdb_id by total_anomaly_count (up to 20):")
    for i, row in enumerate(top_df.itertuples(index=False), 1):
        cmdb = row.cmdb_id
        cnt = int(row.total_anomaly_count)
        ts = row.earliest_anomaly_ts
        ts_s = ts.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if hasattr(ts, 'tzinfo') else str(ts)
        sev = row.max_breach_severity
        if pd.isna(sev):
            sev_s = "N/A"
        elif sev == float('inf'):
            sev_s = "inf"
        else:
            sev_s = f"{float(sev):.6f}"
        lines.append(f"  {i}) {cmdb} — total_anomaly_count={cnt}, earliest={ts_s}, max_breach_severity={sev_s}")
else:
    lines.append("- No trace anomalies detected in the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_anomaly_count       earliest_anomaly_ts  max_breach_severity
2  docker_003                  327 2020-05-30 03:30:00+08:00             0.489842
3  docker_004                  322 2020-05-30 03:30:00+08:00             0.578695
0  docker_001                  319 2020-05-30 03:30:00+08:00             1.964286
1  docker_002                  308 2020-05-30 03:30:00+08:00             0.594113
7  docker_008                  209 2020-05-30 03:30:00+08:00             0.393939
5  docker_006                  208 2020-05-30 03:30:00+08:00             1.978407
6  docker_007                  202 2020-05-30 03:30:00+08:00             2.333333
4  docker_005                  196 2020-05-30 03:30:00+08:00             2.333333
8      os_021                  173 2020-05-30 03:30:00+08:00            15.665717
9      os_022                  168 2020-05-30 03:30:00+08:00             0.552239, 10)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# reuse existing variables if present
tz = pytz.timezone('Asia/Shanghai')
if 'trace_with_thresh' not in globals():
    # ensure trace_df and trace_thresh exist and merge (do NOT recompute thresholds)
    if 'trace_df' not in globals():
        trace_df = pd.read_csv('trace.csv')
    if 'ts_dt' not in trace_df.columns:
        trace_df['ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    if 'trace_thresh' not in globals():
        trace_thresh = (
            trace_df
            .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
            .agg(p95=lambda s: s.quantile(0.95), p5=lambda s: s.quantile(0.05))
        )
    trace_with_thresh = trace_df.merge(trace_thresh, on=['cmdb_id', 'trace_name'], how='left')

# define incident window (inclusive)
start = pd.Timestamp('2020-05-30 03:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-30 04:00:00').tz_localize(tz)

# filter to window and target cmdb_ids
target_cmdbs = ['docker_001','docker_002','docker_003','docker_004']
window_df = trace_with_thresh[
    (trace_with_thresh['ts_dt'] >= start) &
    (trace_with_thresh['ts_dt'] <= end) &
    (trace_with_thresh['cmdb_id'].isin(target_cmdbs))
].copy()

# detect anomaly points using global thresholds (value >= p95 or value <= p5)
anoms = window_df[(window_df['value'] >= window_df['p95']) | (window_df['value'] <= window_df['p5'])].copy()

# compute breach severity per anomaly row
if not anoms.empty:
    high_sev = pd.Series(np.nan, index=anoms.index)
    low_sev = pd.Series(np.nan, index=anoms.index)

    mask_high = anoms['value'] >= anoms['p95']
    denom_high = anoms.loc[mask_high, 'p95']
    num_high = anoms.loc[mask_high, 'value'] - anoms.loc[mask_high, 'p95']
    high_sev.loc[mask_high] = num_high / denom_high.replace(0, np.nan)
    zero_den_high = mask_high & (anoms['p95'] == 0) & (anoms['value'] > anoms['p95'])
    high_sev.loc[zero_den_high] = np.inf

    mask_low = anoms['value'] <= anoms['p5']
    denom_low = anoms.loc[mask_low, 'p5']
    num_low = anoms.loc[mask_low, 'p5'] - anoms.loc[mask_low, 'value']
    low_sev.loc[mask_low] = num_low / denom_low.replace(0, np.nan)
    zero_den_low = mask_low & (anoms['p5'] == 0) & (anoms['value'] < anoms['p5'])
    low_sev.loc[zero_den_low] = np.inf

    anoms['severity'] = pd.concat([high_sev.fillna(-np.inf), low_sev.fillna(-np.inf)], axis=1).max(axis=1)
    anoms.loc[anoms['severity'] == -np.inf, 'severity'] = np.nan
else:
    anoms['severity'] = pd.Series(dtype=float)

# aggregate per (cmdb_id, trace_name)
if not anoms.empty:
    agg = (
        anoms
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('ts_dt', 'min'),
            max_breach_severity=('severity', 'max')
        )
    )
    # round severity for compactness; keep infinities
    agg['max_breach_severity'] = agg['max_breach_severity'].replace(np.inf, np.inf)
    agg['max_breach_severity'] = agg['max_breach_severity'].round(6)
else:
    agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_breach_severity'])

# sort by cmdb_id then anomaly_count desc, keep up to 100 rows, display top 20
agg_sorted = agg.sort_values(['cmdb_id', 'anomaly_count'], ascending=[True, False])
agg_top100 = agg_sorted.head(100)
agg_display = agg_top100.head(20)[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_breach_severity']]

# one-line counts: number of distinct trace_name anomalies per each target cmdb_id
distinct_counts = (
    anoms.groupby('cmdb_id')['trace_name']
    .nunique()
    .reindex(target_cmdbs, fill_value=0)
    .reset_index()
    .rename(columns={'trace_name': 'distinct_trace_name_anomalies'})
)

# Final compact outputs
agg_display, distinct_counts

```
Out[4]:
```
```python
# Compose a plain-English summary string using previous results (agg_display, distinct_counts)
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Ensure variables exist
agg = agg_display if 'agg_display' in globals() else None
dc = distinct_counts if 'distinct_counts' in globals() else None

lines = []
lines.append("Trace anomaly summary for window 2020-05-30 03:30:00+08 to 2020-05-30 04:00:00+08 (inclusive).")
lines.append("Target cmdb_ids: ['docker_001','docker_002','docker_003','docker_004'].")

# distinct counts per cmdb_id
if dc is not None and not dc.empty:
    counts_str = ", ".join(f"{r.cmdb_id}={int(r.distinct_trace_name_anomalies)}" for r in dc.itertuples(index=False))
    lines.append(f"- Distinct trace_name anomalies per target: {counts_str}.")
else:
    lines.append("- No distinct counts available.")

# top examples from agg_display (already limited to top 20 rows)
if agg is not None and not agg.empty:
    lines.append("- Top anomaly rows (showing up to 5 examples from the displayed top 20):")
    for row in agg.head(5).itertuples(index=False):
        cmdb = row.cmdb_id
        tname = row.trace_name
        cnt = int(row.anomaly_count)
        ts = row.earliest_anomaly_timestamp
        ts_s = ts.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if hasattr(ts, 'tzinfo') else str(ts)
        sev = row.max_breach_severity
        if pd.isna(sev):
            sev_s = "N/A"
        elif sev == float('inf'):
            sev_s = "inf"
        else:
            sev_s = f"{float(sev):.6f}"
        lines.append(f"  - {cmdb} | {tname} — count={cnt}, earliest={ts_s}, max_severity={sev_s}")
else:
    lines.append("- No anomalies detected for the requested cmdb_ids in the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                      trace_name  anomaly_count earliest_anomaly_timestamp  max_breach_severity
2   docker_001      trace.from_os_022.remoteprocess.error_rate             30  2020-05-30 03:30:00+08:00                  NaN
10  docker_001                 trace.self.flyremote.error_rate             30  2020-05-30 03:30:00+08:00                  NaN
14  docker_001                      trace.self.jdbc.error_rate             30  2020-05-30 03:30:00+08:00                  NaN
18  docker_001                     trace.self.local.error_rate             30  2020-05-30 03:30:00+08:00                  NaN
22  docker_001    trace.to_docker_007.remoteprocess.error_rate             30  2020-05-30 03:30:00+08:00                  NaN
26  docker_001    trace.to_docker_008.remoteprocess.error_rate             30  2020-05-30 03:30:00+08:00                  NaN
13  docker_001                    trace.self.jdbc.duration_p95             28  2020-05-30 03:30:00+08:00             1.964286
6   docker_001                       trace.self.csf.error_rate             27  2020-05-30 03:30:00+08:00             0.348579
21  docker_001  trace.to_docker_007.remoteprocess.duration_p95              8  2020-05-30 03:31:00+08:00             0.393453
0   docker_001   trace.from_os_022.remoteprocess.duration_mean              4  2020-05-30 03:30:00+08:00             0.106535
1   docker_001    trace.from_os_022.remoteprocess.duration_p95              4  2020-05-30 03:30:00+08:00             0.264631
3   docker_001       trace.from_os_022.remoteprocess.row_count              4  2020-05-30 03:34:00+08:00             0.393939
4   docker_001                    trace.self.csf.duration_mean              4  2020-05-30 03:30:00+08:00             0.531496
5   docker_001                     trace.self.csf.duration_p95              4  2020-05-30 03:30:00+08:00             0.107506
7   docker_001                        trace.self.csf.row_count              4  2020-05-30 03:34:00+08:00             0.501385
8   docker_001              trace.self.flyremote.duration_mean              4  2020-05-30 03:43:00+08:00             0.146437
9   docker_001               trace.self.flyremote.duration_p95              4  2020-05-30 03:43:00+08:00             0.108527
11  docker_001                  trace.self.flyremote.row_count              4  2020-05-30 03:34:00+08:00             0.393939
12  docker_001                   trace.self.jdbc.duration_mean              4  2020-05-30 03:37:00+08:00             0.313865
15  docker_001                       trace.self.jdbc.row_count              4  2020-05-30 03:34:00+08:00             0.642857,       cmdb_id  distinct_trace_name_anomalies
0  docker_001                             28
1  docker_002                             28
2  docker_003                             28
3  docker_004                             28)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# reuse existing variables where possible
tz = pytz.timezone('Asia/Shanghai')

# ensure metric_df with ts_dt exists
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
if 'ts_dt' not in metric_df.columns:
    metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure global thresholds per (cmdb_id, kpi_name) exist (compute from full series if needed)
if 'metric_with_thresh' not in globals():
    group_thresh = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .agg(p95=lambda s: s.quantile(0.95), p5=lambda s: s.quantile(0.05))
    )
    metric_with_thresh = metric_df.merge(group_thresh, on=['cmdb_id', 'kpi_name'], how='left')

# define incident window (inclusive) in UTC+8
start = pd.Timestamp('2020-05-30 03:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-30 04:00:00').tz_localize(tz)

# filter to window
metric_window = metric_with_thresh[(metric_with_thresh['ts_dt'] >= start) & (metric_with_thresh['ts_dt'] <= end)].copy()

# detect anomaly points using global thresholds (value >= p95 or value <= p5)
metric_anoms = metric_window[(metric_window['value'] >= metric_window['p95']) | (metric_window['value'] <= metric_window['p5'])].copy()

# select only db_* cmdb_id
db_anoms = metric_anoms[metric_anoms['cmdb_id'].str.startswith('db_')].copy()

# If no db anomalies, prepare empty outputs
if db_anoms.empty:
    db_agg = pd.DataFrame(columns=[
        'cmdb_id','total_anomaly_count','earliest_anomaly_timestamp','max_breach_severity','top_5_kpis'
    ])
    distinct_db_count_line = "0 distinct db_* components had anomalies in the window."
else:
    # compute per-row severity
    high_sev = pd.Series(np.nan, index=db_anoms.index)
    low_sev = pd.Series(np.nan, index=db_anoms.index)

    mask_high = db_anoms['value'] >= db_anoms['p95']
    denom_high = db_anoms.loc[mask_high, 'p95']
    num_high = db_anoms.loc[mask_high, 'value'] - db_anoms.loc[mask_high, 'p95']
    high_sev.loc[mask_high] = num_high / denom_high.replace(0, np.nan)
    zero_den_high = mask_high & (db_anoms['p95'] == 0) & (db_anoms['value'] > db_anoms['p95'])
    high_sev.loc[zero_den_high] = np.inf

    mask_low = db_anoms['value'] <= db_anoms['p5']
    denom_low = db_anoms.loc[mask_low, 'p5']
    num_low = db_anoms.loc[mask_low, 'p5'] - db_anoms.loc[mask_low, 'value']
    low_sev.loc[mask_low] = num_low / denom_low.replace(0, np.nan)
    zero_den_low = mask_low & (db_anoms['p5'] == 0) & (db_anoms['value'] < db_anoms['p5'])
    low_sev.loc[zero_den_low] = np.inf

    db_anoms['severity'] = pd.concat([high_sev.fillna(-np.inf), low_sev.fillna(-np.inf)], axis=1).max(axis=1)
    db_anoms.loc[db_anoms['severity'] == -np.inf, 'severity'] = np.nan

    # aggregate per db_* cmdb_id
    db_agg_main = (
        db_anoms
        .groupby('cmdb_id', as_index=False)
        .agg(
            total_anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('ts_dt', 'min'),
            max_breach_severity=('severity', 'max')
        )
    )

    # prepare top 5 KPIs by anomaly_count per db
    kpi_counts = (
        db_anoms
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .size()
        .rename(columns={'size':'kpi_anom_count'})
    )

    # for each db, get top 5 KPIs and join as compact string "kpi(count);..."
    def top_kpis_for_db(df):
        top = df.sort_values('kpi_anom_count', ascending=False).head(5)
        return "; ".join(f"{r.kpi_name}({int(r.kpi_anom_count)})" for r in top.itertuples(index=False))

    top_kpis_series = kpi_counts.groupby('cmdb_id').apply(top_kpis_for_db).rename('top_5_kpis').reset_index()

    # merge top kpis into main agg
    db_agg = db_agg_main.merge(top_kpis_series, on='cmdb_id', how='left')

    # round severity for compactness, keep inf
    db_agg['max_breach_severity'] = db_agg['max_breach_severity'].replace(np.inf, np.inf)
    db_agg['max_breach_severity'] = db_agg['max_breach_severity'].round(6)

    # sort by total_anomaly_count desc and limit to top 20
    db_agg = db_agg.sort_values('total_anomaly_count', ascending=False).head(20)

    distinct_db_count_line = f"{db_agg_main['cmdb_id'].nunique()} distinct db_* components had anomalies in the window."

# Final compact outputs (selected columns and bounded)
db_agg[['cmdb_id','total_anomaly_count','earliest_anomaly_timestamp','max_breach_severity','top_5_kpis']], distinct_db_count_line

```
Out[5]:
```
```python
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse db_agg and distinct_db_count_line from the kernel
df = db_agg if 'db_agg' in globals() else None
count_line = distinct_db_count_line if 'distinct_db_count_line' in globals() else None

lines = []
lines.append("Metric anomaly summary for db_* components (window 2020-05-30 03:30:00+08 to 2020-05-30 04:00:00+08):")
if count_line:
    lines.append(f"- {count_line}")
else:
    lines.append("- No summary count available.")

if df is None or df.empty:
    lines.append("- No db_* anomalies found in the window.")
else:
    lines.append("- Top db_* by total_anomaly_count (up to 5 shown):")
    for i, row in enumerate(df.head(5).itertuples(index=False), 1):
        cmdb = row.cmdb_id
        total = int(row.total_anomaly_count)
        ts = row.earliest_anomaly_timestamp
        ts_s = ts.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if hasattr(ts, 'tzinfo') else str(ts)
        sev = row.max_breach_severity
        if pd.isna(sev):
            sev_s = "N/A"
        elif sev == float('inf'):
            sev_s = "inf"
        else:
            sev_s = f"{float(sev):.6f}"
        topk = row.top_5_kpis if hasattr(row, 'top_5_kpis') else ""
        lines.append(f"  {i}) {cmdb} — total_anomaly_count={total}, earliest={ts_s}, max_breach_severity={sev_s}, top_kpis=[{topk}]")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  total_anomaly_count earliest_anomaly_timestamp  max_breach_severity                                         top_5_kpis
5   db_006                  754  2020-05-30 03:30:00+08:00            19.000000  AIOS(30); Hang(30); Sess_Active(30); Row_Lock(...
12  db_013                  667  2020-05-30 03:30:00+08:00            19.000000  ACS(30); AIOS(30); AWS(30); Hang(30); tnsping_...
4   db_005                  635  2020-05-30 03:30:00+08:00             4.150347  ACS(30); AIOS(30); Hang(30); Row_Lock(30); On_...
9   db_010                  626  2020-05-30 03:30:00+08:00           835.470588  ACS(30); AIOS(30); AWS(30); Hang(30); Login_Pe...
10  db_011                  620  2020-05-30 03:30:00+08:00                  inf  AIOS(30); AWS(30); Hang(30); tnsping_result_ti...
7   db_008                  601  2020-05-30 03:30:00+08:00             1.000000  AIOS(30); Hang(30); Proc_User_Used_Pct(30); Ro...
1   db_002                  595  2020-05-30 03:30:00+08:00            82.000000  ACS(30); AIOS(30); Hang(30); Row_Lock(30); Sct...
0   db_001                  589  2020-05-30 03:30:00+08:00             4.333333  ACS(30); AIOS(30); Hang(30); Row_Lock(30); Sct...
11  db_012                  585  2020-05-30 03:30:00+08:00            15.250000  ACS(30); AIOS(30); AWS(30); Hang(30); tnsping_...
3   db_004                  583  2020-05-30 03:30:00+08:00             0.745032  ACS(30); AIOS(30); Hang(30); Row_Lock(30); Sct...
8   db_009                  570  2020-05-30 03:30:00+08:00            22.269948  AIOS(30); DFParaWrite_Per_Sec(30); Hang(30); R...
6   db_007                  568  2020-05-30 03:30:00+08:00             0.394279  ACS(30); AIOS(30); Hang(30); On_Off_State(30);...
2   db_003                  484  2020-05-30 03:30:00+08:00             3.699478  Hang(30); On_Off_State(30); Row_Lock(30); Logi..., '13 distinct db_* components had anomalies in the window.')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed db_anoms if available; otherwise reconstruct from metric_with_thresh and window
if 'db_anoms' not in globals():
    # ensure metric_with_thresh exists
    if 'metric_with_thresh' not in globals():
        metric_df = pd.read_csv('metric.csv')
        metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        group_thresh = (
            metric_df
            .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
            .agg(p95=lambda s: s.quantile(0.95), p5=lambda s: s.quantile(0.05))
        )
        metric_with_thresh = metric_df.merge(group_thresh, on=['cmdb_id', 'kpi_name'], how='left')
    # filter window
    start = pd.Timestamp('2020-05-30 03:30:00').tz_localize(tz)
    end = pd.Timestamp('2020-05-30 04:00:00').tz_localize(tz)
    metric_window = metric_with_thresh[(metric_with_thresh['ts_dt'] >= start) & (metric_with_thresh['ts_dt'] <= end)].copy()
    # detect anomalies
    metric_anoms = metric_window[(metric_window['value'] >= metric_window['p95']) | (metric_window['value'] <= metric_window['p5'])].copy()
    db_anoms = metric_anoms[metric_anoms['cmdb_id'].str.startswith('db_')].copy()

# Ensure per-row severity exists
if 'severity' not in db_anoms.columns or db_anoms['severity'].isnull().all():
    high_sev = pd.Series(np.nan, index=db_anoms.index)
    low_sev = pd.Series(np.nan, index=db_anoms.index)

    mask_high = db_anoms['value'] >= db_anoms['p95']
    denom_high = db_anoms.loc[mask_high, 'p95']
    num_high = db_anoms.loc[mask_high, 'value'] - db_anoms.loc[mask_high, 'p95']
    high_sev.loc[mask_high] = num_high / denom_high.replace(0, np.nan)
    zero_den_high = mask_high & (db_anoms['p95'] == 0) & (db_anoms['value'] > db_anoms['p95'])
    high_sev.loc[zero_den_high] = np.inf

    mask_low = db_anoms['value'] <= db_anoms['p5']
    denom_low = db_anoms.loc[mask_low, 'p5']
    num_low = db_anoms.loc[mask_low, 'p5'] - db_anoms.loc[mask_low, 'value']
    low_sev.loc[mask_low] = num_low / denom_low.replace(0, np.nan)
    zero_den_low = mask_low & (db_anoms['p5'] == 0) & (db_anoms['value'] < db_anoms['p5'])
    low_sev.loc[zero_den_low] = np.inf

    db_anoms['severity'] = pd.concat([high_sev.fillna(-np.inf), low_sev.fillna(-np.inf)], axis=1).max(axis=1)
    db_anoms.loc[db_anoms['severity'] == -np.inf, 'severity'] = np.nan

# 1) Identify top 5 db_* by their max_breach_severity (descending)
db_max_sev = (
    db_anoms
    .groupby('cmdb_id', as_index=False)
    .agg(max_breach_severity=('severity', 'max'))
    .sort_values('max_breach_severity', ascending=False)
)
top5_dbs = db_max_sev.head(5)['cmdb_id'].tolist()

# 2) For these top5, aggregate anomalies per kpi_name
top5_anoms = db_anoms[db_anoms['cmdb_id'].isin(top5_dbs)].copy()

if top5_anoms.empty:
    agg_kpi = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_breach_severity'])
else:
    agg_kpi = (
        top5_anoms
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('ts_dt', 'min'),
            max_breach_severity=('severity', 'max')
        )
    )
    # round severity for compactness, keep inf
    agg_kpi['max_breach_severity'] = agg_kpi['max_breach_severity'].replace(np.inf, np.inf)
    agg_kpi['max_breach_severity'] = agg_kpi['max_breach_severity'].round(6)

# 3) Sort by cmdb_id then anomaly_count desc; limit to top 50, display top 20
agg_sorted = agg_kpi.sort_values(['cmdb_id', 'anomaly_count'], ascending=[True, False])
agg_top50 = agg_sorted.head(50)
agg_display = agg_top50.head(20)[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_breach_severity']]

# 4) Single-line listing of top5 db cmdb_ids in order
top5_line = "Top5 db cmdb_ids by max_breach_severity: " + ", ".join(top5_dbs)

# Final compact outputs (DataFrame and single-line string)
agg_display, top5_line

```
Out[6]:
```
# Prepare a concise plain-English summary and show the aggregated KPI rows (top 20) for the selected top-5 dbs.
import pytz
tz = pytz.timezone('Asia/Shanghai')

# reuse variables from kernel
top5_line = top5_line if 'top5_line' in globals() else None
agg_display = agg_display if 'agg_display' in globals() else None
agg_kpi = agg_kpi if 'agg_kpi' in globals() else None

total_kpi_rows = int(agg_kpi.shape[0]) if agg_kpi is not None else 0
top5_list = top5_line.split(": ",1)[1] if top5_line else "N/A"

summary = (
    f"Summary:\n"
    f"- Top 5 db components by max breach severity (descending): {top5_list}.\n"
    f"- For these top 5 dbs we aggregated anomalies per KPI. Total KPI-rows aggregated for the top-5 set: {total_kpi_rows}.\n"
    f"- Displaying up to 20 rows (columns: cmdb_id, kpi_name, anomaly_count, earliest_anomaly_timestamp, max_breach_severity), sorted by cmdb_id then anomaly_count desc."
)

# Final compact outputs: summary string and the displayed aggregated rows (up to 20)
summary, agg_display

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id               kpi_name  anomaly_count earliest_anomaly_timestamp  max_breach_severity
0   db_002                    ACS             30  2020-05-30 03:30:00+08:00             0.500000
1   db_002                   AIOS             30  2020-05-30 03:30:00+08:00                  NaN
10  db_002                   Hang             30  2020-05-30 03:30:00+08:00             0.000000
14  db_002          Login_Per_Sec             30  2020-05-30 03:30:00+08:00             5.000000
15  db_002              MEM_Total             30  2020-05-30 03:30:00+08:00             0.000000
21  db_002           On_Off_State             30  2020-05-30 03:30:00+08:00             0.000000
25  db_002          Proc_Used_Pct             30  2020-05-30 03:30:00+08:00             0.000000
26  db_002     Proc_User_Used_Pct             30  2020-05-30 03:30:00+08:00             0.000000
28  db_002               Row_Lock             30  2020-05-30 03:30:00+08:00                  NaN
29  db_002        SctRead_Per_Sec             30  2020-05-30 03:30:00+08:00                  NaN
42  db_002    tnsping_result_time             30  2020-05-30 03:30:00+08:00             0.000000
24  db_002  Physical_Read_Per_Sec             28  2020-05-30 03:30:00+08:00            82.000000
2   db_002                    AWS             27  2020-05-30 03:30:00+08:00             0.000000
36  db_002            TPS_Per_Sec             25  2020-05-30 03:30:00+08:00             0.058824
22  db_002           PGA_Used_Pct             15  2020-05-30 03:31:00+08:00                  NaN
23  db_002         PGA_used_total             15  2020-05-30 03:31:00+08:00             0.000000
31  db_002            Sess_Active             15  2020-05-30 03:31:00+08:00             1.000000
33  db_002         Sess_Used_Temp             15  2020-05-30 03:31:00+08:00                  NaN
34  db_002         Sess_Used_Undo             15  2020-05-30 03:31:00+08:00                  NaN
39  db_002            TempTbs_Pct             15  2020-05-30 03:31:00+08:00                  NaN, 'Top5 db cmdb_ids by max_breach_severity: db_011, db_010, db_002, db_009, db_006')```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# reuse existing metric_df and tz if available
tz = pytz.timezone('Asia/Shanghai')
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
if 'ts_dt' not in metric_df.columns:
    metric_df['ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 for every kpi_name belonging to db_011 using full series
db_id = 'db_011'
db011_full = metric_df[metric_df['cmdb_id'] == db_id].copy()

# selected KPIs of interest (at least these)
selected_kpis = ['Hang', 'tnsping_result_time', 'ACS', 'AIOS', 'AWS']

# compute thresholds for all KPIs of db_011, then filter to selected_kpis that actually exist
if not db011_full.empty:
    thresholds_all = (
        db011_full
        .groupby('kpi_name', as_index=False)['value']
        .agg(global_P95=lambda s: s.quantile(0.95), global_P5=lambda s: s.quantile(0.05))
    )
    # keep only selected kpis that exist for db_011
    existing_kpis = thresholds_all[thresholds_all['kpi_name'].isin(selected_kpis)].copy()
    # round thresholds for compact display
    existing_kpis['global_P95'] = existing_kpis['global_P95'].round(6)
    existing_kpis['global_P5'] = existing_kpis['global_P5'].round(6)
else:
    existing_kpis = pd.DataFrame(columns=['kpi_name','global_P95','global_P5'])

# 2) Filter metric.csv to the incident window and select rows for db_011 and the selected KPIs (intersection)
start = pd.Timestamp('2020-05-30 03:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-30 04:00:00').tz_localize(tz)

kpis_to_fetch = existing_kpis['kpi_name'].tolist()  # intersection only
window_rows = db011_full[
    (db011_full['ts_dt'] >= start) &
    (db011_full['ts_dt'] <= end) &
    (db011_full['kpi_name'].isin(kpis_to_fetch))
].copy()

# keep only needed columns and sort by timestamp ascending
window_rows = window_rows[['ts_dt', 'kpi_name', 'value']].sort_values('ts_dt', ascending=True).head(50)

# 3) Determine earliest anomalous timestamp in the window for these KPIs using the precomputed global thresholds
if not window_rows.empty and not existing_kpis.empty:
    # merge thresholds onto window rows by kpi_name
    window_with_thresh = window_rows.merge(existing_kpis.rename(columns={'kpi_name':'kpi_name'}), on='kpi_name', how='left')
    # detect anomaly where value >= P95 or value <= P5
    is_anom = (window_with_thresh['value'] >= window_with_thresh['global_P95']) | (window_with_thresh['value'] <= window_with_thresh['global_P5'])
    anom_rows = window_with_thresh[is_anom].copy()
    if not anom_rows.empty:
        earliest_anom_ts = anom_rows['ts_dt'].min()
        earliest_anom_line = f"Earliest anomaly for {db_id} (selected KPIs) in window: {earliest_anom_ts.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')}"
    else:
        earliest_anom_ts = None
        earliest_anom_line = f"No anomalies for {db_id} among selected KPIs in the window."
else:
    anom_rows = pd.DataFrame(columns=['ts_dt','kpi_name','value','global_P95','global_P5'])
    earliest_anom_ts = None
    earliest_anom_line = f"No data for {db_id} and selected KPIs in the window."

# Compact outputs:
#  - existing_kpis: DataFrame with kpi_name, global_P95, global_P5 (only selected & existing)
#  - window_rows: up to 50 metric rows in the window with (timestamp tz-aware, kpi_name, value)
#  - earliest_anom_line: one-line earliest anomaly timestamp or message

existing_kpis[['kpi_name','global_P95','global_P5']], window_rows, earliest_anom_line

```
Out[7]:
```
```python
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse variables computed in the kernel: existing_kpis, window_rows, earliest_anom_line
thresholds_df = existing_kpis[['kpi_name','global_P95','global_P5']].copy() if 'existing_kpis' in globals() else None
window_rows_sorted = window_rows.sort_values('ts_dt', ascending=True).head(50) if 'window_rows' in globals() else None
earliest_line = earliest_anom_line if 'earliest_anom_line' in globals() else "No earliest anomaly info available."

# Compact outputs: thresholds per KPI, up to 50 metric rows in the window, and earliest anomaly line
thresholds_df, window_rows_sorted, earliest_line
```

The original code execution output of IPython Kernel is also provided below for reference:

(               kpi_name  global_P95  global_P5
0                   ACS      0.0105        0.0
1                  AIOS      0.0000        0.0
2                   AWS      0.0100        0.0
10                 Hang      1.0000        1.0
44  tnsping_result_time     20.0000       10.0,                           ts_dt             kpi_name  value
45921 2020-05-30 03:30:00+08:00                  ACS   0.01
45922 2020-05-30 03:30:00+08:00                 AIOS   0.00
45923 2020-05-30 03:30:00+08:00                  AWS   0.00
45930 2020-05-30 03:30:00+08:00                 Hang   1.00
45952 2020-05-30 03:30:00+08:00  tnsping_result_time  10.00
...                         ...                  ...    ...
59483 2020-05-30 03:39:00+08:00                  ACS   0.01
59484 2020-05-30 03:39:00+08:00                 AIOS   0.00
59485 2020-05-30 03:39:00+08:00                  AWS   0.01
59491 2020-05-30 03:39:00+08:00                 Hang   1.00
59518 2020-05-30 03:39:00+08:00  tnsping_result_time  20.00

[50 rows x 3 columns], 'Earliest anomaly for db_011 (selected KPIs) in window: 2020-05-30 03:30:00+0800')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```